In [70]:
! pip install pytorch-lightning wandb rdkit ogb
!pip install torch_geometric

In [71]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/My Drive/courses_@IITD/ML4Chem_CML7211/solubility_data.csv")
df.head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1
5,benzothiazole,-2.733,2,135.191,0,2,0,12.89,-1.50,c2ccc1scnc1c2
6,"2,2,4,6,6'-PCB",-6.545,1,326.437,0,2,1,0.00,-7.32,Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl
7,Estradiol,-4.138,1,272.388,2,4,0,40.46,-5.03,CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O
8,Dieldrin,-4.533,1,380.913,0,5,0,12.53,-6.29,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
9,Rotenone,-5.246,1,394.423,0,5,3,63.22,-4.42,COc5cc4OCC3Oc2c1CC(Oc1ccc2C(=O)C3c4cc5OC)C(C)=C


In [72]:
smiles_list = df['smiles']
targets = df['measured log solubility in mols per litre']

In [73]:
from rdkit import Chem

valid_smiles = []
valid_targets = []

for smi, y in zip(smiles_list, targets):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_smiles.append(smi)
        valid_targets.append(y)

print("Valid molecules:", len(valid_smiles))

Valid molecules: 1128


In [74]:
import torch
from torch_geometric.data import Data

def mol_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)

    # --- Node features ---
    atom_fvs = []
    for atom in mol.GetAtoms():
        atom_fvs.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            atom.GetNumImplicitHs(),
            int(atom.GetIsAromatic())
        ])

    # --- Edges ---
    edge_index = []
    bond_fvs = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index += [[i, j], [j, i]]

        bf = [
            bond.GetBondTypeAsDouble(),
            int(bond.GetIsConjugated()),
            int(bond.IsInRing())
        ]

        bond_fvs += [bf, bf]

    # --- Convert ---
    x = torch.tensor(atom_fvs, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bond_fvs, dtype=torch.float)
    #x = torch.tensor(atom_fvs, dtype=torch.float)
    #edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    #edge_attr = torch.tensor(bond_fvs, dtype=torch.float)
    y = torch.tensor([[y]], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.smiles = smiles

    return data

In [75]:
dataset = [mol_to_graph(smi, y) for smi, y in zip(valid_smiles, valid_targets)]

In [76]:
def normalize_dataset(dataset):
    y_all = torch.cat([data.y for data in dataset], dim=0)
    mean, std = y_all.mean(), y_all.std()

    for data in dataset:
        data.y = (data.y - mean) / std

    return dataset, mean.item(), std.item()

In [77]:
dataset, mean, std = normalize_dataset(dataset)

In [78]:
import random

#dataset = dataset.shuffle()
random.shuffle(dataset)

n = len(dataset)
train_dataset = dataset[:int(0.7*n)]
val_dataset   = dataset[int(0.7*n):int(0.8*n)]
test_dataset  = dataset[int(0.8*n):]

In [79]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

##MPNN

In [80]:
import torch.nn as nn
from torch_geometric.nn import NNConv, global_mean_pool

class MPNNModel(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_dim=64):
        super().__init__()

        # --- Edge network for conv1 ---
        nn1 = nn.Sequential(
            nn.Linear(edge_dim, 128),
            nn.ReLU(),
            nn.Linear(128, in_channels * hidden_dim)
        )

        # --- Edge network for conv2 ---
        nn2 = nn.Sequential(
            nn.Linear(edge_dim, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_dim * hidden_dim)
        )

        self.conv1 = NNConv(in_channels, hidden_dim, nn1, aggr='mean')
        self.conv2 = NNConv(hidden_dim, hidden_dim, nn2, aggr='mean')

        self.lin = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)

## Initialize

In [81]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MPNNModel(
    in_channels=train_dataset[0].num_node_features,
    edge_dim=train_dataset[0].num_edge_features
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

## Training

In [82]:
def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)

        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)

        loss = loss_fn(out, data.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

## Validation

In [83]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = loss_fn(out, data.y)

        total_loss += loss.item()

    return total_loss / len(loader)

In [84]:
for epoch in range(1, 101):
    train_loss = train()
    val_loss = evaluate(val_loader)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Train {train_loss:.4f}, Val {val_loss:.4f}")

Epoch 10: Train 0.7204, Val 0.7423
Epoch 20: Train 0.5749, Val 0.5735
Epoch 30: Train 0.4589, Val 0.4559
Epoch 40: Train 0.4361, Val 0.9142
Epoch 50: Train 0.3697, Val 0.3858
Epoch 60: Train 0.2842, Val 0.4540
Epoch 70: Train 0.2672, Val 0.3045
Epoch 80: Train 0.2843, Val 0.3779
Epoch 90: Train 0.2220, Val 0.2530
Epoch 100: Train 0.2393, Val 0.3577


## Evaluate

In [85]:
import torch.nn.functional as F

def denormalize(y, mean, std):
    return y * std + mean

@torch.no_grad()
def test_rmse():
    model.eval()

    y_true_all = []
    y_pred_all = []

    for data in test_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)

        y_pred = denormalize(out, mean, std)
        y_true = denormalize(data.y, mean, std)

        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

    y_true_all = torch.cat(y_true_all)
    y_pred_all = torch.cat(y_pred_all)

    rmse = torch.sqrt(F.mse_loss(y_pred_all, y_true_all))
    return rmse.item()

In [86]:
rmse = test_rmse()
print(f"Test RMSE: {rmse:.3f} logS")

Test RMSE: 1.152 logS


## Try out printing some log S values

In [105]:
import torch

@torch.no_grad()
def print_smiles_predictions(model, dataset, smiles_list, mean, std, device, num_samples=10):
    model.eval()

    for i in range(num_samples):
        data = dataset[i].to(device)

        out = model(data.x, data.edge_index, data.edge_attr,
                    torch.zeros(data.x.size(0), dtype=torch.long).to(device))  # single graph

        # Denormalize
        y_pred = out * std + mean
        y_true = data.y * std + mean

        print(f"SMILES: {smiles_list[i]}")
        print(f"Predicted logS: {y_pred.item():.3f}")
        print(f"True logS:      {y_true.item():.3f}")
        print("-"*50)

In [106]:
print_smiles_predictions(model, test_dataset, smiles_list, mean, std, device)

SMILES: OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)C(O)C3O 
Predicted logS: -4.349
True logS:      -4.030
--------------------------------------------------
SMILES: Cc1occc1C(=O)Nc2ccccc2
Predicted logS: -1.932
True logS:      -3.220
--------------------------------------------------
SMILES: CC(C)=CCCC(C)=CC(=O)
Predicted logS: -4.531
True logS:      -2.640
--------------------------------------------------
SMILES: c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
Predicted logS: -3.282
True logS:      -4.300
--------------------------------------------------
SMILES: c1ccsc1
Predicted logS: -8.479
True logS:      -9.150
--------------------------------------------------
SMILES: c2ccc1scnc1c2 
Predicted logS: -1.704
True logS:      0.070
--------------------------------------------------
SMILES: Clc1cc(Cl)c(c(Cl)c1)c2c(Cl)cccc2Cl
Predicted logS: -2.926
True logS:      -3.080
--------------------------------------------------
SMILES: CC12CCC3C(CCc4cc(O)ccc34)C2CCC1O
Predicted logS: -5.478
True logS: